# 05 — Interactive Gradio Demo

This notebook wraps the full retrieval pipeline in a **Gradio** web UI so anyone can try it without touching code.

**Features:**
- Text input for the search query
- Dropdown to choose retrieval model: `GloVe`, `MiniLM`, or `MiniLM + Cross-encoder Reranker`
- Slider for how many tweets to return (top-k)
- Output: retrieved tweets + FLAN-T5 generated summary

**Deployment:** The last cell launches a public link via `share=True` — paste it into HuggingFace Spaces or share directly.

**Prerequisites:** Run `01_build_index.ipynb` first.

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
!pip install gradio faiss-cpu sentence-transformers transformers torch --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
import torch
import gradio as gr
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
# --- STEP 2: LOAD INDEXES AND TWEETS ---

base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS indexes...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
glove_index  = faiss.read_index(f'{base}glove_faiss.index')

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

print(f"MiniLM index: {minilm_index.ntotal:,} vectors")
print(f"GloVe index:  {glove_index.ntotal:,} vectors")
print(f"Tweets loaded: {len(tweets):,}")

In [ ]:
# --- STEP 3: LOAD MODELS ---

print("Loading MiniLM bi-encoder...")
minilm_encoder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading GloVe encoder...")
glove_encoder = SentenceTransformer('average_word_embeddings_glove.840B.300d')

print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Loading FLAN-T5 for summarization...")
try:
    flan_name = 'google/flan-t5-large'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_name).to(device)
    print("FLAN-T5-large loaded.")
except RuntimeError:
    # Fall back to base if large doesn't fit in memory
    print("FLAN-T5-large OOM — loading flan-t5-base instead.")
    flan_name = 'google/flan-t5-base'
    flan_tokenizer = T5Tokenizer.from_pretrained(flan_name)
    flan_model = T5ForConditionalGeneration.from_pretrained(flan_name).to(device)
    print("FLAN-T5-base loaded.")

print("\nAll models ready!")

In [ ]:
# --- STEP 4: PIPELINE FUNCTIONS ---

def retrieve_minilm(query, top_k):
    """
    Retrieve top-k tweets using MiniLM + FAISS cosine search.

    Parameters:
        query (str): Search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by similarity (best first).
    """
    vec = minilm_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    vec = vec / np.linalg.norm(vec)  # normalize for cosine similarity via inner product
    _, indices = minilm_index.search(vec, top_k)
    return indices[0].tolist()


def retrieve_glove(query, top_k):
    """
    Retrieve top-k tweets using GloVe averaged embeddings + FAISS L2 search.

    Parameters:
        query (str): Search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by L2 distance (closest first).
    """
    vec = glove_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    _, indices = glove_index.search(vec, top_k)
    return indices[0].tolist()


def rerank(query, tweet_indices, top_n):
    """
    Rerank candidate tweets with a cross-encoder and return the top-n.

    Parameters:
        query (str): Original search query.
        tweet_indices (list[int]): Candidate tweet indices from FAISS retrieval.
        top_n (int): How many to keep after reranking.

    Returns:
        list[int]: Top-n tweet indices in reranked order.
    """
    pairs = [[query, tweets[idx]] for idx in tweet_indices]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(tweet_indices, scores), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked[:top_n]]


def generate_summary(query, tweet_indices):
    """
    Generate a FLAN-T5 summary of the top retrieved tweets.

    Parameters:
        query (str): The original search query.
        tweet_indices (list[int]): Tweet indices to use as context (top 5 used).

    Returns:
        str: Natural language summary generated by FLAN-T5.
    """
    context = [tweets[i] for i in tweet_indices[:5]]
    numbered = '\n'.join(f"{i+1}. {t}" for i, t in enumerate(context))
    prompt = (
        f"Context tweets:\n{numbered}\n\n"
        f"Based on the tweets above, summarize what people are saying about: {query}"
    )
    inputs = flan_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    out = flan_model.generate(**inputs, max_new_tokens=200, num_beams=4)
    return flan_tokenizer.decode(out[0], skip_special_tokens=True)


print("Pipeline functions defined.")

In [ ]:
# --- STEP 5: GRADIO HANDLER ---

def search_handler(query, model_choice, top_k):
    """
    Main Gradio handler — runs the selected pipeline and returns formatted results.

    Parameters:
        query (str): User's search query from the text box.
        model_choice (str): One of 'GloVe', 'MiniLM', or 'MiniLM + Reranker'.
        top_k (int): Number of tweets to display (from the slider).

    Returns:
        tuple[str, str]: (formatted tweet results, FLAN-T5 summary)
    """
    if not query.strip():
        return "Please enter a query.", ""

    top_k = int(top_k)

    if model_choice == 'GloVe':
        indices = retrieve_glove(query, top_k)

    elif model_choice == 'MiniLM':
        indices = retrieve_minilm(query, top_k)

    else:  # MiniLM + Reranker
        # Retrieve a larger pool first so the reranker has more candidates to work with
        candidates = retrieve_minilm(query, top_k=50)
        indices = rerank(query, candidates, top_n=top_k)

    summary = generate_summary(query, indices)

    # Format tweet list for display
    tweet_lines = []
    for rank, idx in enumerate(indices, start=1):
        tweet_lines.append(f"{rank}. {tweets[idx]}")
    tweet_display = "\n\n".join(tweet_lines)

    return tweet_display, summary


print("Gradio handler defined.")

In [ ]:
# --- STEP 6: LAUNCH GRADIO INTERFACE ---

demo = gr.Interface(
    fn=search_handler,
    inputs=[
        gr.Textbox(
            label="Search Query",
            placeholder="e.g. looking for a job",
            lines=1
        ),
        gr.Dropdown(
            choices=['GloVe', 'MiniLM', 'MiniLM + Reranker'],
            value='MiniLM + Reranker',
            label="Retrieval Model"
        ),
        gr.Slider(
            minimum=5,
            maximum=25,
            value=10,
            step=1,
            label="Number of tweets (top-k)"
        )
    ],
    outputs=[
        gr.Textbox(label="Retrieved Tweets", lines=15),
        gr.Textbox(label="FLAN-T5 Summary", lines=5)
    ],
    title="SpatialSearch — Tweet RAG Pipeline",
    description=(
        "Search 110,000+ tweets using semantic embeddings. "
        "Choose between GloVe (static embeddings), MiniLM (contextual BERT-based), "
        "or MiniLM + a cross-encoder reranker for highest accuracy. "
        "FLAN-T5 summarizes the top results."
    ),
    examples=[
        ["looking for a job",       "MiniLM + Reranker", 10],
        ["climate change",           "MiniLM",            10],
        ["feeling happy today",      "GloVe",             10],
    ],
    allow_flagging='never'
)

# share=True creates a public tunnel URL (works on Colab and local)
# Set share=False to run locally only
demo.launch(share=True)